In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install transformers>=4.30.0
!pip install accelerate
!pip install nltk
# !conda install pytorch torchvision torchaudio cpuonly -c pytorch
!pip install transformers>=4.30.0 accelerate nltk

import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import torch
import time
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize

class Timer:
    def __init__(self):
        self.start = time.time()
    def stop(self):
        elapsed = time.time() - self.start
        print(f" Elapsed: {elapsed:.2f}s")
        return elapsed

model_path = "amphora/FinABSA"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# NER pipeline for AUTO-DETECTING ALL aspects (companies + financial terms)
ner_pipeline = pipeline("ner", 
                       model="Jean-Baptiste/roberta-large-ner-english", 
                       aggregation_strategy="simple")

def extract_aspects_dynamic(text):
    aspects = []
    
    # 1. NER detects companies/organizations automatically
    entities = ner_pipeline(text)
    
    # 2. Extract ALL entity types (ORG, MISC, LOC if financial-relevant)
    for ent in entities:
        if ent['score'] > 0.8:  # High confidence only
            aspects.append(ent['word'])
    
    # 3. Capitalized nouns as potential financial aspects (heuristic)
    words = word_tokenize(text)
    cap_nouns = [w for w in words if w[0].isupper() and len(w) > 3 and not w.endswith(('s', "'s"))]
    aspects.extend([noun for noun in cap_nouns if noun not in aspects])
    
    # 4. Deduplicate
    aspects = list(set(aspects))
    
    return aspects if aspects else ["company"]  # Fallback

def run_finabsa(text):
    mentioned_aspects = extract_aspects_dynamic(text)
    
    results = {}
    for aspect in mentioned_aspects[:5]:  # Limit to top 5 to avoid noise
        if aspect == "company" or len(aspect) < 2:
            continue
            
        input_str = text.replace(aspect, "[TGT]", 1)
        inputs = tokenizer(input_str, return_tensors='pt', truncation=True, max_length=512)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=30, num_beams=1, do_sample=False)
        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        sentiment = "NEGATIVE" if "NEGATIVE" in prediction else "POSITIVE" if "POSITIVE" in prediction else "NEUTRAL"
        results[aspect] = sentiment
        
    return results if results else {"No aspects detected": "NEUTRAL"}

examples = [
    "Nvidia reports record GPU sales but warns of AI chip supply constraints",
    "Amazon cloud division AWS beats expectations amid enterprise AI migration",
    "Bitcoin surges past $95K as MicroStrategy adds 10K BTC to holdings",
    "Boeing shares plunge 8% after FAA grounds 737 MAX fleet following incidents",
    "Goldman Sachs raises S&P 500 target to 6200 on earnings momentum",
    "Ethereum ETF inflows hit $2B while Solana developers face network congestion issues",
    "Meta advertising revenue grows 25% but Reality Labs posts $4B losses",
    "ExxonMobil beats oil production targets despite OPEC+ cuts"
]

print(" FULLY DYNAMIC ABSA (NER + linguistic heuristics)")
for i, text in enumerate(examples):
    print(f"\n\n### EXAMPLE {i} ###")
    print(f"Input: {text}")
    results = run_finabsa(text)
    print("Detected Aspects & ABSA:")
    for aspect, sentiment in results.items():
        print(f"  {aspect}: {sentiment}")


Looking in indexes: https://download.pytorch.org/whl/cpu


Device set to use cpu


🔍 FULLY DYNAMIC ABSA (NER + linguistic heuristics)


### EXAMPLE 0 ###
Input: Nvidia reports record GPU sales but warns of AI chip supply constraints
Detected Aspects & ABSA:
  Nvidia: NEUTRAL
   Nvidia: NEUTRAL


### EXAMPLE 1 ###
Input: Amazon cloud division AWS beats expectations amid enterprise AI migration
Detected Aspects & ABSA:
   Amazon: POSITIVE
  Amazon: POSITIVE
   AWS: POSITIVE


### EXAMPLE 2 ###
Input: Bitcoin surges past $95K as MicroStrategy adds 10K BTC to holdings
Detected Aspects & ABSA:
  MicroStrategy: NEUTRAL
   MicroStrategy: NEUTRAL
  Bitcoin: POSITIVE


### EXAMPLE 3 ###
Input: Boeing shares plunge 8% after FAA grounds 737 MAX fleet following incidents
Detected Aspects & ABSA:
   FAA: NEUTRAL
   737 MAX: NEUTRAL
   Boeing: NEUTRAL
  Boeing: NEGATIVE


### EXAMPLE 4 ###
Input: Goldman Sachs raises S&P 500 target to 6200 on earnings momentum
Detected Aspects & ABSA:
   S&P 500: POSITIVE
   Goldman Sachs: POSITIVE
  Goldman: NEUTRAL


### EXAMPLE 5 ###
Input: Eth